<a href="https://colab.research.google.com/github/Shaithra2107/HALT-RAG/blob/lift_signal/HALT_RAG_v5_scan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HALT-RAG v5 — Corpus scan (run this BEFORE v5_signals)

Two problems with the original scan, and why they block everything else.

**Problem 1 — it stopped early.** The scan read the first 3,000,000 of KILT's
~5.9 million pages. 164 of your 300 gold pages were never reached. For those
questions "was the answer covered?" really means "we never looked there", so
every number computed from them is partly an artefact of where the scan stopped.

**Problem 2 — the distractors were not a sample.** The old code kept the *first*
500 eligible pages in the stream. KILT streams in a fixed order, so that is an
arbitrary slice of Wikipedia, not a random sample of it. Lift is defined as
*"how far above what this collection returns at random"* — and you cannot
estimate that from an arbitrary slice, no matter how many you take.

This notebook does a full pass and uses **reservoir sampling**, which gives
every eligible page an equal chance of ending up in the sample, in one pass,
without knowing the stream length in advance.

**Time:** roughly 30–60 minutes for the full pass. It checkpoints every 200,000
records, so you can disconnect and re-run this cell to resume.

**Outputs:** `kilt_cache_v5.pkl` and `halt_rag_audit_v5.csv`. Point v5_signals
at those.

## Cell 1 — Install

In [ ]:
!pip show datasets | head -2
import datasets, sys
print("imported :", datasets.__version__)
print("loaded from:", datasets.__file__)

Name: datasets
Version: 3.6.0
ERROR: Pipe to stdout was broken
imported : 3.6.0
loaded from: /usr/local/lib/python3.13/dist-packages/datasets/__init__.py


In [ ]:
!pip install -q "datasets==3.6.0" tqdm pandas numpy 2>&1 | tail -3
import datasets, sys
print("datasets", datasets.__version__, "| python", sys.version.split()[0])

datasets 3.6.0 | python 3.13.16


In [ ]:
!pip install -q "datasets==3.6.0"

## Cell 2 — Drive + CONFIG

In [ ]:
import os, json
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/halt_rag_cache'
os.makedirs(DRIVE_DIR, exist_ok=True)

CONFIG = {
    'audit_in'       : f'{DRIVE_DIR}/halt_rag_audit_full.csv',   # your existing 300 rows
    'audit_out'      : f'{DRIVE_DIR}/halt_rag_audit_v5.csv',     # same rows, M2 recomputed
    'cache_out'      : f'{DRIVE_DIR}/kilt_cache_v5.pkl',
    'ckpt'           : f'{DRIVE_DIR}/kilt_ckpt_v5.pkl',

    # None = full pass over KILT. Set an integer only to test the plumbing.
    'scan_limit'     : None,

    # FIX 2 — reservoir size. These are the pages Lift's baseline is drawn from.
    # 10k pages is roughly 1-3 GB of text once chunked; drop to 5000 if Colab
    # runs out of RAM, raise it if you have headroom.
    'reservoir'      : 10_000,
    'max_para_per_distractor': 40,   # guard: stops one huge page eating the RAM

    'checkpoint_every': 200_000,
    'min_para_chars' : 50,
    'seed'           : 42,
}
print(json.dumps(CONFIG, indent=2))
assert os.path.exists(CONFIG['audit_in']), f"put halt_rag_audit_full.csv in {DRIVE_DIR}/"

Mounted at /content/drive
{
  "audit_in": "/content/drive/MyDrive/halt_rag_cache/halt_rag_audit_full.csv",
  "audit_out": "/content/drive/MyDrive/halt_rag_cache/halt_rag_audit_v5.csv",
  "cache_out": "/content/drive/MyDrive/halt_rag_cache/kilt_cache_v5.pkl",
  "ckpt": "/content/drive/MyDrive/halt_rag_cache/kilt_ckpt_v5.pkl",
  "scan_limit": null,
  "reservoir": 10000,
  "max_para_per_distractor": 40,
  "checkpoint_every": 200000,
  "min_para_chars": 50,
  "seed": 42
}


## Cell 3 — Target titles, taken from your existing audit CSV

Deliberately **not** re-derived from PopQA. If the dataset version or the
sampling seed ever drifted you would silently be scanning for a different set of
300 entities, and nothing downstream would line up. Reading the titles from the
CSV you already built removes that risk entirely.

In [ ]:
import pandas as pd, numpy as np, re, ast

def normalise_title(t): return re.sub(r'_', ' ', str(t).lower().strip())
def normalise_text(t):  return re.sub(r'\s+', ' ', str(t).lower().strip())

def parse_answers(val):
    if val is None or (isinstance(val, float) and pd.isna(val)): return []
    if isinstance(val, list): ans = val
    else:
        try: ans = ast.literal_eval(str(val))
        except Exception: ans = [str(val)]
    return [str(a).strip() for a in ans if str(a).strip() and str(a).strip().lower() != 'nan']

audit = pd.read_csv(CONFIG['audit_in'])
audit['answers_list_p'] = audit['answers_list'].apply(parse_answers)

target_norm2orig = {}
for _, r in audit.iterrows():
    target_norm2orig[normalise_title(r['wiki_title'])] = r['wiki_title']

print(f"audit rows        : {len(audit)}")
print(f"unique target titles: {len(target_norm2orig)}")
print(f"band split        : {audit['band'].value_counts().to_dict()}")
print(f"rows with no parsed answers: {int((audit['answers_list_p'].apply(len)==0).sum())}")
print()
print("previous M2 status (what we are trying to improve):")
print(audit['m2_status'].value_counts().to_string())

audit rows        : 300
unique target titles: 299
band split        : {'tail': 150, 'head': 150}
rows with no parsed answers: 0

previous M2 status (what we are trying to improve):
m2_status
not_in_kilt_scan    164
found               120
not_found            16


## Cell 4 — The scan

Keeps two things as it streams:

* every page whose title is one of your 300 targets
* a **reservoir** of `CONFIG['reservoir']` randomly chosen non-target pages

Reservoir sampling, Algorithm R: once the reservoir is full, page number *n*
replaces a random existing slot with probability `reservoir / n`. Every eligible
page therefore ends up in the sample with the same probability, which is exactly
what Lift's baseline requires.

Checkpoints carry the reservoir, the counter and the RNG state, so resuming
gives the same sample you would have got in one run.

In [ ]:
import pickle, time
from itertools import islice
from datasets import load_dataset
from tqdm.auto import tqdm

kilt_pages, reservoir = {}, []
scanned, n_eligible, iter_state = 0, 0, None
rng = np.random.default_rng(CONFIG['seed'])

if os.path.exists(CONFIG['cache_out']):
    print(f"completed cache already exists: {CONFIG['cache_out']}")
    print("delete it if you want to re-scan from scratch.")
    with open(CONFIG['cache_out'], 'rb') as f: _c = pickle.load(f)
    kilt_pages, reservoir = _c['kilt_pages'], _c['distractor_pool']
    scanned, n_eligible = _c['scanned'], _c.get('n_eligible', len(reservoir))
else:
    if os.path.exists(CONFIG['ckpt']):
        with open(CONFIG['ckpt'], 'rb') as f: _c = pickle.load(f)
        kilt_pages  = _c['kilt_pages'];  reservoir = _c['distractor_pool']
        scanned     = _c['scanned'];     n_eligible = _c['n_eligible']
        iter_state  = _c.get('iter_state')
        if _c.get('rng_state') is not None:
            rng.bit_generator.state = _c['rng_state']
        print(f"resuming from {scanned:,} records "
              f"({len(kilt_pages)} targets found, reservoir {len(reservoir)})")
    else:
        print("starting a fresh full pass ...")

    ds = load_dataset("facebook/kilt_wikipedia", split="full",
                      streaming=True, trust_remote_code=True)
    it = iter(ds)

    if iter_state is not None:
        try:
            it.load_state_dict(iter_state)
            print("iterator state restored")
        except Exception as e:
            print(f"state restore failed ({e}); fast-forwarding {scanned:,} records")
            for _ in tqdm(islice(it, scanned), total=scanned, desc="fast-forward"): pass

    R      = CONFIG['reservoir']
    LIMIT  = CONFIG['scan_limit']
    t0     = time.time()

    for krow in tqdm(it, desc="scanning KILT"):
        if LIMIT is not None and scanned >= LIMIT: break
        scanned += 1

        norm  = normalise_title(krow['wikipedia_title'])
        paras = krow['text']['paragraph'] if isinstance(krow['text'], dict) else []
        paras = [p for p in paras if isinstance(p, str)
                 and len(p.strip()) > CONFIG['min_para_chars']]
        if not paras:
            continue

        if norm in target_norm2orig:
            kilt_pages[norm] = paras                     # keep every target page in full
        else:
            n_eligible += 1
            trimmed = paras[:CONFIG['max_para_per_distractor']]
            if len(reservoir) < R:
                reservoir.append({'title': krow['wikipedia_title'], 'paragraphs': trimmed})
            else:
                j = int(rng.integers(0, n_eligible))     # Algorithm R
                if j < R:
                    reservoir[j] = {'title': krow['wikipedia_title'], 'paragraphs': trimmed}

        if scanned % CONFIG['checkpoint_every'] == 0:
            ck = {'kilt_pages': kilt_pages, 'distractor_pool': reservoir,
                  'scanned': scanned, 'n_eligible': n_eligible,
                  'rng_state': rng.bit_generator.state}
            try: ck['iter_state'] = it.state_dict()
            except Exception: pass
            with open(CONFIG['ckpt'], 'wb') as f: pickle.dump(ck, f)
            rate = scanned / max(time.time() - t0, 1e-9)
            print(f"  checkpoint {scanned:,} | targets {len(kilt_pages)}/{len(target_norm2orig)}"
                  f" | reservoir {len(reservoir)} | {rate:,.0f} rec/s")

    with open(CONFIG['cache_out'], 'wb') as f:
        pickle.dump({'kilt_pages': kilt_pages, 'distractor_pool': reservoir,
                     'scanned': scanned, 'n_eligible': n_eligible,
                     'reservoir_size': CONFIG['reservoir'],
                     'scan_limit': CONFIG['scan_limit']}, f)
    if os.path.exists(CONFIG['ckpt']): os.remove(CONFIG['ckpt'])
    print(f"\nsaved {CONFIG['cache_out']}")

print(f"\nscanned           : {scanned:,}")
print(f"target pages found: {len(kilt_pages)} / {len(target_norm2orig)}")
print(f"eligible non-targets seen: {n_eligible:,}")
print(f"reservoir         : {len(reservoir):,} pages "
      f"({len(reservoir)/max(n_eligible,1)*100:.3f}% of eligible — a uniform sample)")
missing = set(target_norm2orig) - set(kilt_pages)
print(f"still not found   : {len(missing)}")
if missing:
    print("  e.g.", [target_norm2orig[k] for k in list(missing)[:5]])
    print("  these are genuinely absent from KILT 2019, or their title differs in it.")

resuming from 5,000,000 records (133 targets found, reservoir 10000)


scanning KILT: 0it [00:00, ?it/s]

  checkpoint 5,200,000 | targets 133/299 | reservoir 10000 | 7,756 rec/s
  checkpoint 5,400,000 | targets 133/299 | reservoir 10000 | 3,922 rec/s
  checkpoint 5,600,000 | targets 133/299 | reservoir 10000 | 2,698 rec/s
  checkpoint 5,800,000 | targets 133/299 | reservoir 10000 | 2,084 rec/s
  checkpoint 6,000,000 | targets 133/299 | reservoir 10000 | 1,722 rec/s
  checkpoint 6,200,000 | targets 133/299 | reservoir 10000 | 1,484 rec/s


KeyboardInterrupt: 

In [ ]:
import pickle, os
D = '/content/drive/MyDrive/halt_rag_cache'

pickle.dump({'kilt_pages': kilt_pages,
             'distractor_pool': reservoir,
             'scanned': scanned,
             'n_eligible': n_eligible,
             'reservoir_size': len(reservoir),
             'scan_limit': 'stopped_early_no_new_targets'},
            open(f'{D}/kilt_cache_v5.pkl', 'wb'))

print('targets  :', len(kilt_pages))
print('reservoir:', len(reservoir))
print('scanned  :', f'{scanned:,}')

targets  : 133
reservoir: 10000
scanned  : 6,300,338


## Cell 5 — Recompute M2 and write the updated audit CSV

M2 asks: does the answer string appear on the subject's own KILT page? With a
full pass, far fewer questions should come back `not_in_kilt_scan`.

A page that is still missing after a full pass is a real finding, not a gap in
the method: that entity has no page in the 2019 snapshot under that title.
Report those as genuinely uncovered rather than unknown.

In [ ]:
def answer_in_text(answers, text_norm):
    for a in answers:
        a = normalise_text(a)
        if not a: continue
        if re.search(r'(?<![\w])' + re.escape(a) + r'(?![\w])', text_norm):
            return True
    return False

m2_match, m2_status = [], []
for _, r in audit.iterrows():
    key = normalise_title(r['wiki_title'])
    if key not in kilt_pages:
        m2_match.append(np.nan); m2_status.append('not_in_kilt'); continue
    if not r['answers_list_p']:
        m2_match.append(np.nan); m2_status.append('no_valid_answers'); continue
    body = normalise_text(' '.join(kilt_pages[key]))
    hit  = answer_in_text(r['answers_list_p'], body)
    m2_match.append(int(hit)); m2_status.append('found' if hit else 'not_found')

out = audit.drop(columns=['answers_list_p']).copy()
out['m2_match_v5']  = m2_match
out['m2_status_v5'] = m2_status

print("M2 status BEFORE (3M partial scan):")
print(audit['m2_status'].value_counts().to_string())
print("\nM2 status AFTER (full pass):")
print(out['m2_status_v5'].value_counts().to_string())

ev_before = audit['m2_status'].isin(['found','not_found']).sum()
ev_after  = out['m2_status_v5'].isin(['found','not_found']).sum()
print(f"\nquestions with a KNOWN coverage answer: {ev_before} -> {ev_after}  "
      f"(+{ev_after-ev_before})")
print("by band:")
print(pd.crosstab(out['m2_status_v5'], out['band']).to_string())

out.to_csv(CONFIG['audit_out'], index=False)
print(f"\nsaved {CONFIG['audit_out']}")

M2 status BEFORE (3M partial scan):
m2_status
not_in_kilt_scan    164
found               120
not_found            16

M2 status AFTER (full pass):
m2_status_v5
not_in_kilt    166
found          120
not_found       14

questions with a KNOWN coverage answer: 136 -> 134  (+-2)
by band:
band          head  tail
m2_status_v5            
found           67    53
not_found        5     9
not_in_kilt     78    88

saved /content/drive/MyDrive/halt_rag_cache/halt_rag_audit_v5.csv


## Cell 6 — Sanity check the reservoir

Two things worth confirming before you trust Lift.

**Is it actually a sample?** The share of eligible pages kept should be close to
`reservoir / n_eligible`, and the titles should look like arbitrary Wikipedia,
not an alphabetical run — which is what the old first-500 approach produced.

**Is it big enough?** Lift divides by the standard deviation of the baseline
similarities. With too few passages that standard deviation is itself noisy, and
Lift inherits the noise.

In [ ]:
print(f"reservoir {len(reservoir):,} pages from {n_eligible:,} eligible "
      f"({len(reservoir)/max(n_eligible,1)*100:.3f}%)")
print("\n20 random reservoir titles — these should look unrelated to each other:")
for t in rng.choice([d['title'] for d in reservoir],
                    size=min(20, len(reservoir)), replace=False):
    print("  ", t)

first_letters = pd.Series([str(d['title'])[:1].upper() for d in reservoir])
print("\nfirst-letter spread (a real sample is spread out; the old first-500")
print("approach clustered heavily because KILT streams in a fixed order):")
print(first_letters.value_counts().head(12).to_string())

n_para = pd.Series([len(d['paragraphs']) for d in reservoir])
print(f"\nparagraphs per reservoir page: median {n_para.median():.0f}, "
      f"mean {n_para.mean():.1f}, capped at {CONFIG['max_para_per_distractor']}")
print(f"approx passages after 100-word chunking: {int(n_para.sum()*1.2):,}")

reservoir 10,000 pages from 6,233,072 eligible (0.160%)

20 random reservoir titles — these should look unrelated to each other:
   John Cannon
   John Kay (economist)
   2014–15 FC Steaua București season
   Telephone numbers in Curaçao and the Caribbean Netherlands
   Drift Allstars
   Culture of Jordan
   List of Japanese films of 2007
   Aceramarca gracile opossum
   Members of the Australian Senate, 1965–1968
   Luigi Squarzina
   Canadian Women's Suffrage Association
   Laurent Angliviel de la Beaumelle
   Reynolds (crater)
   FIS Alpine World Ski Championships 2017 – Men's alpine combined
   Melbourne Bowling Club
   Trip-a-Tron
   Kindia Region
   Nadi Rural (Indian Communal Constituency, Fiji)
   Garra flavatra
   Otis M. Whitney

first-letter spread (a real sample is spread out; the old first-500
approach clustered heavily because KILT streams in a fixed order):
S    818
C    691
A    687
M    666
T    609
L    574
B    561
P    499
J    451
R    413
D    408
H    380

paragr

In [ ]:
import pickle, os
D = '/content/drive/MyDrive/halt_rag_cache'

ck = pickle.load(open(f'{D}/kilt_ckpt_v5.pkl', 'rb'))
print('targets  :', len(ck['kilt_pages']))
print('reservoir:', len(ck['distractor_pool']))
print('scanned  :', f"{ck['scanned']:,}")

pickle.dump({'kilt_pages'     : ck['kilt_pages'],
             'distractor_pool': ck['distractor_pool'],
             'scanned'        : ck['scanned'],
             'n_eligible'     : ck.get('n_eligible'),
             'reservoir_size' : len(ck['distractor_pool']),
             'scan_limit'     : 'stopped_early_no_new_targets'},
            open(f'{D}/kilt_cache_v5.pkl', 'wb'))

kilt_pages = ck['kilt_pages']      # so cell 5 can use it
print('\nsaved kilt_cache_v5.pkl')

targets  : 133
reservoir: 10000
scanned  : 6,200,000

saved kilt_cache_v5.pkl


---

## Next

In **v5_signals**, change cell 2 to point at the new files:

```python
'audit_csv' : f'{DRIVE_DIR}/halt_rag_audit_v5.csv',
```

and cell 3 will pick up `kilt_cache_v5.pkl` automatically.

Then in v5_signals use `m2_status_v5` wherever the old `m2_status` was used, so
the `evaluable` flag reflects the full pass.

### What this notebook does and does not fix

**Fixes:** the 164 unknown coverage labels, and Lift's baseline — which becomes
a genuine random sample of the collection for the first time.

**Does not fix:** `y_cov` is still the answer-string match your own audit showed
is wrong on 57% of its positives. That needs the published Joren et al. autorater
prompt. It is the next thing to do after this runs.